# Рубежный контроль 1 · **вариант 8**

**Группа:** ИУ5-65Б *(позиция в списке = 8 → вариант 8).*

**Доп. требование для ИУ5-65Б:** построить **парные диаграммы** (*pair plot*) для набора данных.

**Задача 1.** Корреляционный анализ; при пропусках — удалить строки или столбцы с пропусками; выводы о пригодности данных для моделей машинного обучения и о вкладе признаков.

**Датасет:** Boston Housing Dataset ([Kaggle: altavish/boston-housing-dataset](https://www.kaggle.com/datasets/altavish/boston-housing-dataset)).


## 1. Загрузка данных

Сначала пробуем загрузку через **kagglehub** (если Kaggle уже настроен). Иначе — резервная ссылка на CSV (структура как у классического Boston Housing: 13 признаков + `MEDV`).


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df = None
try:
    import kagglehub
    folder = Path(kagglehub.dataset_download("altavish/boston-housing-dataset"))
    csvs = sorted(folder.glob("*.csv"))
    if not csvs:
        csvs = sorted(folder.rglob("*.csv"))
    df = pd.read_csv(csvs[0])
    print(f"Загружено из kagglehub: {csvs[0].name}")
except Exception as e1:
    print("kagglehub:", e1)
    urls = [
        "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv",
        "https://gist.githubusercontent.com/nguyen-toan/867bb45c87d6bcf3bdc8eae9e14f0636/raw/"
        "BostonHousing/BostonHousing.csv",
    ]
    for url in urls:
        try:
            df = pd.read_csv(url)
            print("Загружено по URL:", url)
            break
        except Exception as e2:
            print("URL failed:", url[:60], "...", repr(e2))
    if df is None:
        raise SystemExit(
            "Не удалось загрузить данные. Установите kagglehub или скачайте CSV с Kaggle вручную в папку с ноутбуком."
        )

# Имена столбцов в едином регистре (в выгрузках Kaggle/GitHub часто lower case)
df.columns = [str(c).upper() for c in df.columns]
if "MEDV" not in df.columns:
    raise ValueError("Не найден целевой столбец MEDV — проверьте CSV.")
print("Столбцы:", list(df.columns))
print(df.shape)
df.head()


## 2. Приведение к числовому виду и искусственные пропуски

По **примечанию к РК:** если пропусков нет, часть значений можно заменить на `NaN` для демонстрации шага очистки. Здесь случайно «обнуляем» ~6% ячеек в столбцах **`AGE`** и **`RM`**.


In [ ]:
FEATURES_ORIG = df.copy()

# Дополнительный категориальный признак (по примеру из методички)
# RAD — индекс доступности магистралей → «зона»: низкая / высокая
if "RAD" in df.columns:
    df["_rad_zone"] = np.where(df["RAD"].astype(float) <= df["RAD"].astype(float).median(), "Низкая", "Высокая")
else:
    df["_rad_zone"] = "Неизвестно"

rng = np.random.default_rng(8)  # вариант 8 — фиксированный seed
for col in ["AGE", "RM"]:
    if col not in df.columns:
        continue
    vc = df[col].astype(float).copy()
    mask = rng.random(len(df)) < 0.06
    vc.loc[mask] = np.nan
    df[col] = vc

print("Пропуски до удаления строк:")
print(df.isna().sum().sort_values(ascending=False))
print("\nСтрок до:", len(df))

# Задача: удалить строки или столбцы с пропусками — удаляем **строки**
df_clean = df.dropna(axis=0, how="any").copy()
removed = len(df) - len(df_clean)
print("Удалено строк с NaN:", removed, "| Осталось:", len(df_clean))


## 3. Корреляционный анализ

**Коэффициенты корреляции Пирсона** для числовых признаков; тепловая карта. Категория **`_rad_zone`** в корреляции не участвует (будет на pairplot).


In [ ]:
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
if "RAD" in num_cols and "_rad_zone" not in df_clean.columns:
    pass

corr_p = df_clean[num_cols].corr(method="pearson")
corr_s = df_clean[num_cols].corr(method="spearman")

print("Pearson corr с MEDV (по модулю убывание):\n")
print(corr_p["MEDV"].drop("MEDV").abs().sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(corr_p, ax=axes[0], annot=False, cmap="RdBu_r", center=0, linewidths=0.2)
axes[0].set_title("Корреляция Пирсона")

sns.heatmap(corr_s, ax=axes[1], annot=False, cmap="RdBu_r", center=0, linewidths=0.2)
axes[1].set_title("Корреляция Спирмена")

plt.tight_layout()
plt.show()


## 4. Требование ИУ5-65Б: **парные диаграммы** (`pairplot`)

Подмножество признаков: сильнее всего связанные с **`MEDV`** и смежные показатели. Оттенок (`hue`) — зона доступности дорог (**`_rad_zone`**). Если график слишком тяжёлый, можно уменьшить число столбцов.


In [ ]:
# Несколько главных признаков + MEDV для читаемости
cands = [
    "RM", "LSTAT", "PTRATIO", "INDUS", "NOX", "DIS", "TAX",
    "AGE", "CRIM", "RAD", "MEDV",
]

pair_cols = [c for c in cands if c in df_clean.columns]
cols_pair = []
for c in ["RM", "LSTAT", "PTRATIO", "INDUS", "NOX", "MEDV"]:
    if c in pair_cols:
        cols_pair.append(c)

sub = df_clean[cols_pair + (["_rad_zone"] if "_rad_zone" in df_clean.columns else [])].copy()

hue_col = "_rad_zone" if "_rad_zone" in sub.columns else None

g = sns.pairplot(sub, hue=hue_col, corner=True, plot_kws=dict(alpha=0.65, s=18))
g.figure.suptitle("Парные диаграммы (ИУ5-65Б)", y=1.02)
plt.show()


## 5. Выводы для задачи 1

Заполните по своей интерпретации после запуска (ориентиры):

1. **Пропуски:** после генерации случайных `NaN` и удаления строк сохранился репрезентативный объём данных; альтернативно можно было удалить «проблемный» столбец вместо строк — зависит от доли потерь.
2. **Корреляции с MEDV:** обычно заметную **положительную** связь с **`RM`** (число комнат) и **отрицательную** с **`LSTAT`**, **`PTRATIO`**, загрязнение и др. Конкретные значения см. столбцы `corr_p["MEDV"]` после запуска.
3. **Построение моделей:** при умеренных линейных корреляциях уместны **линейные** и **регуляризованные** регрессии; при нелинейных паттернах на pairplot — **деревья**, **ансамбли**, **SVR** и т.д. Сильно коррелирующие между собой признаки (многоколлинеарность) могут требовать регуляризации или отбора признаков.
4. **Вклад признаков:** признаки с высоким |corr| с целевым потенциально важны; пары с |corr|≈1 между собой избыточны. Окончательный вклад лучше оценивать в обученной модели (коэффициенты, важности), но корреляционный анализ даёт первичный отбор.
